# 第八课：数据为王 —— AI的「食材」管理

## 学习目标
- 理解数据在 AI 工程中的核心地位
- 用代码识别和清理「脏数据」
- 体验用 AI 合成训练数据
- 理解数据增强与质量评估的基本概念

> 垃圾进，垃圾出（Garbage In, Garbage Out）。无论模型多强，数据质量决定一切。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：识别和清理「脏数据」

### 活动目标
拿到一份数据集后，第一件事不是直接使用，而是检查数据质量。
这个活动让你亲手识别数据中的常见问题：重复、矛盾、格式不一致、缺失值。

In [ ]:
# 活动一：识别脏数据

import re

# 模拟一份「客户反馈数据集」
raw_data = [
    {'id': 1, 'name': '张三', 'rating': 5, 'comment': '非常满意！产品很好用。', 'date': '2025-01-15'},
    {'id': 2, 'name': '李四', 'rating': 4, 'comment': '还不错，有点小问题', 'date': '2025-01-15'},
    {'id': 3, 'name': '张三', 'rating': 5, 'comment': '非常满意！产品很好用。', 'date': '2025-01-15'},  # 重复！
    {'id': 4, 'name': '王五', 'rating': 1, 'comment': '太棒了，强烈推荐！', 'date': '2025-01-16'},  # 矛盾！低分+好评
    {'id': 5, 'name': '赵六', 'rating': 3, 'comment': '', 'date': '2025/01/16'},  # 空评论 + 日期格式不一致
    {'id': 6, 'name': '', 'rating': 4, 'comment': '物有所值', 'date': '2025-01-17'},  # 名字缺失
    {'id': 7, 'name': '孙七', 'rating': 99, 'comment': '还行吧', 'date': '2025-01-17'},  # 评分超出范围！
    {'id': 8, 'name': '周八', 'rating': 4, 'comment': 'good product', 'date': '2025-01-18'},  # 语言不一致
]

print('原始数据集：共', len(raw_data), '条记录\n')

# 检查问题
issues = []

# 1. 检查重复
seen = set()
for item in raw_data:
    key = (item['name'], item['comment'])
    if key in seen:
        issues.append(f'ID {item["id"]}: 重复记录（名称+评论与之前相同）')
    seen.add(key)

# 2. 检查评分异常
for item in raw_data:
    if item['rating'] < 1 or item['rating'] > 5:
        issues.append(f'ID {item["id"]}: 评分 {item["rating"]} 超出1-5范围')
    if item['rating'] <= 2 and any(w in item['comment'] for w in ['好', '棒', '推荐', '满意']):
        issues.append(f'ID {item["id"]}: 评分与评论矛盾（低分+好评）')

# 3. 检查缺失值
for item in raw_data:
    if not item['name']:
        issues.append(f'ID {item["id"]}: 姓名为空')
    if not item['comment']:
        issues.append(f'ID {item["id"]}: 评论为空')

# 4. 检查格式与语言一致性
for item in raw_data:
    if '/' in item['date']:
        issues.append(f'ID {item["id"]}: 日期格式不一致（{item["date"]}，其余为 2025-01-15 格式）')
    if item['comment'] and re.fullmatch(r'[\x00-\x7F]+', item['comment']):
        issues.append(f'ID {item["id"]}: 评论语言不一致（非中文：{item["comment"]}）')

print('发现的问题：')
for issue in issues:
    print(f'  - {issue}')

print(f'\n共发现 {len(issues)} 个问题！')
print('如果这些数据用来训练AI，这些问题会导致：')
print('  重复 → 模型过拟合某些样本')
print('  矛盾 → 模型学到混乱的信号')
print('  缺失 → 训练数据不完整')
print('  格式不一致 → 数据处理出错')

### 讨论
- 你平时用的数据有没有类似的问题？
- 如果数据量很大（几万条），怎么发现这些问题？
- 「清洗数据」和「写代码调模型」哪个更耗时？

---

## 活动二：用 AI 合成训练数据

### 活动目标
AI 可以帮你快速生成大量训练数据。但 AI 生成的数据也需要人工审核——质量把控是人的责任。

In [ ]:
# 活动二：AI 合成训练数据

# 场景：为「餐厅客户投诉处理AI」生成训练数据

data_gen_prompt = '''你是一位数据标注专家。请生成10条模拟的餐厅客户投诉。

要求：
1. 覆盖不同类型的投诉：菜品质量、服务态度、上菜速度、环境卫生、价格问题
2. 情绪要有变化：从温和不满到非常愤怒
3. 每条投诉包含：
   - 投诉内容（客户原始表述）
   - 情绪类别（温和/中等/愤怒）
   - 期望解决方式

格式：
投诉内容: ...
情绪: ...
期望: ...'''

r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':data_gen_prompt}],
    temperature=0.8)  # 高温度增加多样性

print('AI生成的数据：\n')
print(r.choices[0].message.content)

print('\n' + '='*50)
print('数据质量检查清单：')
print('1. 是否覆盖了所有要求的投诉类型？')
print('2. 是否有不合理的投诉？（太假、太夸张、不符合常识）')
print('3. 是否有偏见？（如对某种菜品/人群的刻板印象）')
print('4. 情绪和投诉内容是否匹配？')
print('5. 期望解决方式是否合理？')

### 讨论
- AI 生成的数据质量如何？有没有需要修改的？
- 「AI 生成数据 + 人工审核」vs「完全人工标注」——各有什么优劣？
- 如果 AI 生成的数据中有偏见，后续会发生什么？

---

## 活动三：数据增强——用小数据「变」出大数据

### 活动目标
数据增强（Data Augmentation）是用已有数据创造更多变体。
对于文本数据，常见的增强方式包括：同义词替换、句式改写、翻译后回译。

In [ ]:
# 活动三：文本数据增强

original = '这个产品的质量非常好，我很满意，强烈推荐给大家！'

# 增强一：同义词替换
print('=== 增强一：同义词替换 ===')
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'将以下句子中的形容词替换为同义词，保持原意不变：\n{original}'}],
    temperature=0.5)
print(f'原句：{original}')
print(f'变体：{r1.choices[0].message.content}')

# 增强二：句式改写
print('=== 增强二：句式改写 ===')
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'用不同的句式表达以下意思，保持含义不变：\n{original}'}],
    temperature=0.5)
print(f'原句：{original}')
print(f'变体：{r2.choices[0].message.content}')

# 增强三：翻译后回译
print('=== 增强三：翻译后回译 ===')
# 先翻译成英文
r3_en = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'Translate to English: {original}'}],
    temperature=0.3)
english = r3_en.choices[0].message.content
# 再翻译回中文
r3_cn = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'Translate to Chinese: {english}'}],
    temperature=0.3)
print(f'原句：{original}')
print(f'英文：{english}')
print(f'回译：{r3_cn.choices[0].message.content}')

print('一条数据变成了三条——这就是数据增强的力量！')

### 讨论
- 三种增强方式各有什么优缺点？
- 数据增强会不会「增强」出错误的数据？
- 什么时候该用数据增强？什么时候不该用？

---

## 本节回顾

| 技能 | 说明 |
|------|------|
| 脏数据识别 | 发现重复、矛盾、缺失、格式不一致等问题 |
| AI 合成数据 | 用 AI 快速生成训练数据并检查质量 |
| 数据增强 | 用同义词替换、改写、回译等方式扩充数据 |

### 课后练习
1. 找一份你日常使用的表格数据，检查它是否存在脏数据问题
2. 访问 huggingface.co/datasets，浏览3个感兴趣的数据集
3. 用 AI 为你的工作/学习场景生成10条训练数据，并人工审核质量